In [1]:
import numpy as np 
import pandas as pd 
import os
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
import category_encoders as ce

In [ ]:
filepath = "../Train/task2/introml_2024_task2_train.csv"
df = pd.read_csv(filepath)
# 前處理
cols = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'class'] # 0/1變數

for c in cols:
    df[c] = df[c].apply(lambda x: int(x[-1]))
df

df.insert(len(df.columns)-1, 'f18', df['f17'] * df['f13'])
df.insert(len(df.columns)-1, 'f19', df['f17'] * df['f14'])


from sklearn.decomposition import PCA

pca = PCA(n_components=1)
df.insert(len(df.columns)-1, 'f20', pca.fit_transform(df[['f13', 'f14']]))
df.insert(len(df.columns)-1, 'f21', pca.fit_transform(df[['f10', 'f11']]))
# df = df.drop(['f13', 'f14'], axis = 1)
df.insert(len(df.columns)-1, 'f22', df['f6'] + df['f7'])
df.insert(len(df.columns)-1, 'f23', df['f3'] - df['f8']) 
df.insert(len(df.columns)-1, 'f24', df['f3'] - df['f6']) 
# df = df.drop(['f0', 'f1', 'f2', 'f5'], axis=1)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f16,f17,f18,f19,f20,f21,f22,f23,f24,class
0,0,1,0,0,0,0,1,1,1,0.3839,...,1.8718,0.7433,1.685581,0.844463,0.794527,0.455046,2,-1,-1,0
1,0,0,0,0,0,0,1,0,1,1.4089,...,1.5515,0.7560,0.685314,0.930938,-0.545688,0.703835,1,-1,-1,0
2,1,0,0,0,1,0,0,1,0,0.2752,...,2.5584,2.0897,3.587597,2.741268,0.268423,-0.769513,1,0,0,0
3,1,1,0,1,1,0,0,0,0,0.1777,...,0.1408,2.0033,4.402452,1.410323,0.673551,-0.702621,0,1,1,0
4,0,1,0,0,0,0,0,1,0,0.1692,...,0.6857,0.6148,0.996345,0.388492,0.092072,-0.560943,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,1,0,0,0,1,0,1,1,0,0.3347,...,0.6958,0.6273,0.324251,0.465017,-0.990793,-0.337867,2,0,-1,4
3996,0,0,0,0,1,0,1,0,0,0.2989,...,0.3491,0.8280,0.220993,0.961722,-1.189062,-0.450812,1,0,-1,4
3997,0,0,0,1,1,0,1,1,0,0.5245,...,0.3595,1.6582,0.301129,0.914829,-1.346257,-0.343527,2,1,0,4
3998,0,0,0,0,1,0,1,0,0,0.7995,...,0.5401,1.0678,1.113075,0.435769,-0.508635,-0.799093,1,0,-1,4


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate, Lambda
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# 假設 df 已經是 pandas DataFrame
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# 將目標變數進行 one-hot encoding
y = to_categorical(y)

# 分割資料集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 對特徵進行標準化（可選）
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 定義輸入層
input_layer = Input(shape=(X_train.shape[1],))

# 使用 Lambda 層分割特徵
sigmoid_features = Lambda(lambda x: x[:, :9])(input_layer)  # 前 9 個特徵
relu_features = Lambda(lambda x: x[:, 9:])(input_layer)    # 後面特徵

# 為前 9 個特徵添加 sigmoid 激活層
sigmoid_branch = Dense(16, activation='relu')(sigmoid_features)

# 為其他特徵添加 ReLU 激活層
relu_branch = Dense(16, activation='tanh')(relu_features)

# 合併兩個分支
merged = Concatenate()([sigmoid_branch, relu_branch])

# 添加全連接層
hidden = Dense(32, activation='relu')(merged)
output_layer = Dense(y_train.shape[1], activation='softmax')(hidden)  # 輸出層

# 定義模型
model = Model(inputs=input_layer, outputs=output_layer)

# 編譯模型
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 訓練模型
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=2
)

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Accuracy: {test_accuracy:.4f}")


Epoch 1/50
100/100 - 1s - 8ms/step - accuracy: 0.3606 - loss: 1.5104 - val_accuracy: 0.5663 - val_loss: 1.2186
Epoch 2/50
100/100 - 0s - 1ms/step - accuracy: 0.6672 - loss: 1.0059 - val_accuracy: 0.6988 - val_loss: 0.8676
Epoch 3/50
100/100 - 0s - 1ms/step - accuracy: 0.7291 - loss: 0.7856 - val_accuracy: 0.7237 - val_loss: 0.7348
Epoch 4/50
100/100 - 0s - 1ms/step - accuracy: 0.7450 - loss: 0.7026 - val_accuracy: 0.7500 - val_loss: 0.6886
Epoch 5/50
100/100 - 0s - 1ms/step - accuracy: 0.7597 - loss: 0.6693 - val_accuracy: 0.7563 - val_loss: 0.6626
Epoch 6/50
100/100 - 0s - 1ms/step - accuracy: 0.7666 - loss: 0.6497 - val_accuracy: 0.7550 - val_loss: 0.6514
Epoch 7/50
100/100 - 0s - 1ms/step - accuracy: 0.7681 - loss: 0.6356 - val_accuracy: 0.7675 - val_loss: 0.6444
Epoch 8/50
100/100 - 0s - 1ms/step - accuracy: 0.7697 - loss: 0.6259 - val_accuracy: 0.7613 - val_loss: 0.6394
Epoch 9/50
100/100 - 0s - 1ms/step - accuracy: 0.7725 - loss: 0.6165 - val_accuracy: 0.7600 - val_loss: 0.6337
E